In [ ]:
import os
import json
import requests
from pathlib import Path
from urllib.parse import urljoin

VIDEOS_BASE_URL = "https://ai-generated-videos-icl.s3.us-east-1.amazonaws.com/"
ANNOTATIONS_URL = "https://geneva-annotations.s3.us-east-1.amazonaws.com/AnnotatedVideos.json"

OUTPUT_DIR = Path("downloaded_videos")
DOWNLOAD_VIDEOS = True


def download_json(url: str):
    response = requests.get(url)
    response.raise_for_status()
    return response.json()


def download_file(url: str, output_path: Path):
    response = requests.get(url, stream=True)
    response.raise_for_status()

    output_path.parent.mkdir(parents=True, exist_ok=True)

    with open(output_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)


annotations = download_json(ANNOTATIONS_URL)

print(f"Loaded {len(annotations)} annotation entries")
print("Example entry:\n", annotations[0])

for entry in annotations:
    video_name = (
        entry.get("videoID")
    )

    if not video_name:
        continue

    video_url = urljoin(VIDEOS_BASE_URL, video_name)
    print(f"Video URL: {video_url}")

    if DOWNLOAD_VIDEOS:
        output_path = OUTPUT_DIR / video_name
        download_file(video_url, output_path)

